# Sequential Pose Optimization Models

This notebook implements three sequential models for pose optimization using MediaPipe pose landmarks. The models will:
1. Learn correct pose sequences from expert demonstrations
2. Predict optimal pose corrections
3. Generate feedback on pose alignment and form

The models will help provide real-time feedback for workout form correction.

In [1]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import pickle

# Download the dataset
path = kagglehub.dataset_download("hasyimabdillah/workoutfitness-video")
print("Path to dataset files:", path)

# Import our custom modules
import sys
sys.path.append('..')
from mediapipe_handler.sequence_processor import SequenceProcessor

# Create sequence processor for both input and target sequences
sequence_processor = SequenceProcessor(sequence_length=90)  # 3 seconds at 30fps

Path to dataset files: /Users/yasinetawfeek/.cache/kagglehub/datasets/hasyimabdillah/workoutfitness-video/versions/5


I0000 00:00:1759719277.019066 19491714 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1759719277.075303 19492326 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1759719277.085711 19492335 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [3]:
# Process videos and create input-target pairs
def create_pose_optimization_data(video_path, expert_threshold=0.9):
    """
    Create input-target pairs for pose optimization.
    Expert threshold determines which sequences are considered as target poses.
    """
    try:
        with open('../mediapipe_handler/sequences_labels_tuple.pkl', 'rb') as f:
            print("Loading sequences and labels from pickle file...")
            sequences, labels = pickle.load(f)

    except FileNotFoundError:
        print("Pickle file not found. Processing video directory...")
        sequences, labels = sequence_processor.process_video_directory(video_path)
    
    # Assume sequences with higher confidence scores are from expert demonstrations
    confidence_scores = np.random.uniform(0, 1, len(sequences))  # Replace with actual confidence calculation
    
    expert_sequences = sequences[confidence_scores > expert_threshold]
    input_sequences = sequences[confidence_scores <= expert_threshold]
    
    # For each input sequence, find the closest expert sequence as target
    X = []
    y = []
    
    for input_seq in input_sequences:
        distances = np.mean(np.square(expert_sequences - input_seq), axis=(1, 2))
        closest_expert = expert_sequences[np.argmin(distances)]
        
        X.append(input_seq)
        y.append(closest_expert)
    
    return np.array(X), np.array(y)

# Create training data
X, y = create_pose_optimization_data(path)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

print("Training set shape:", X_train.shape)
print("Validation set shape:", X_val.shape)
print("Test set shape:", X_test.shape)

Loading sequences and labels from pickle file...
Training set shape: (15395, 60, 36)
Validation set shape: (1924, 60, 36)
Test set shape: (1925, 60, 36)


## Model Definitions

For pose optimization, we'll create sequence-to-sequence models that can generate corrected pose sequences:

In [4]:
def positional_encoding(length, depth):
    depth = depth / 2
    positions = np.arange(length)[:, np.newaxis]
    depths = np.arange(depth)[np.newaxis, :] / depth
    angle_rates = 1 / (10000 ** depths)
    angle_rads = positions * angle_rates
    pos_encoding = np.concatenate(
        [np.sin(angle_rads), np.cos(angle_rads)],
        axis=-1
    )
    return tf.cast(pos_encoding, dtype=tf.float32)

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation="relu"),
            tf.keras.layers.Dense(embed_dim),
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

In [5]:
# Define seq2seq models for pose optimization
def create_lstm_seq2seq(input_shape):
    # Encoder
    encoder_inputs = tf.keras.Input(shape=input_shape)
    encoder_lstm = tf.keras.layers.LSTM(128, return_sequences=True, return_state=True)
    encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)
    
    # Decoder
    decoder_lstm = tf.keras.layers.LSTM(128, return_sequences=True)
    decoder_outputs = decoder_lstm(encoder_outputs, initial_state=[state_h, state_c])
    
    # Dense output layers
    decoder_dense = tf.keras.layers.Dense(input_shape[-1], activation='linear')
    outputs = decoder_dense(decoder_outputs)
    
    return tf.keras.Model(encoder_inputs, outputs)

def create_gru_seq2seq(input_shape):
    # Encoder
    encoder_inputs = tf.keras.Input(shape=input_shape)
    encoder_gru = tf.keras.layers.GRU(128, return_sequences=True, return_state=True)
    encoder_outputs, state = encoder_gru(encoder_inputs)
    
    # Decoder
    decoder_gru = tf.keras.layers.GRU(128, return_sequences=True)
    decoder_outputs = decoder_gru(encoder_outputs, initial_state=state)
    
    # Dense output layers
    decoder_dense = tf.keras.layers.Dense(input_shape[-1], activation='linear')
    outputs = decoder_dense(decoder_outputs)
    
    return tf.keras.Model(encoder_inputs, outputs)

def create_transformer_seq2seq(input_shape):
    inputs = tf.keras.Input(shape=input_shape)
    
    # Add positional encoding
    pos_encoding = positional_encoding(input_shape[0], input_shape[1])
    x = inputs + pos_encoding
    
    # Transformer blocks
    transformer_block1 = TransformerBlock(input_shape[1], 8, 32)
    transformer_block2 = TransformerBlock(input_shape[1], 8, 32)
    
    x = transformer_block1(x)
    x = transformer_block2(x)
    
    # Output projection
    outputs = tf.keras.layers.Dense(input_shape[-1], activation='linear')(x)
    
    return tf.keras.Model(inputs=inputs, outputs=outputs)

# Create and compile models
lstm_model = create_lstm_seq2seq((X_train.shape[1], X_train.shape[2]))
gru_model = create_gru_seq2seq((X_train.shape[1], X_train.shape[2]))
transformer_model = create_transformer_seq2seq((X_train.shape[1], X_train.shape[2]))

# Custom loss function for pose optimization
def pose_mse_loss(y_true, y_pred):
    """MSE loss with higher weights for key joint positions"""
    # Define weights for different joints (can be customized)
    joint_weights = tf.constant([1.5 if i % 3 == 0 else 1.0 for i in range(y_true.shape[-1])])
    squared_diff = tf.square(y_true - y_pred)
    weighted_squared_diff = squared_diff * joint_weights
    return tf.reduce_mean(weighted_squared_diff)

# Compile models
for model in [lstm_model, gru_model, transformer_model]:
    model.compile(
        optimizer='adam',
        loss=pose_mse_loss,
        metrics=['mse']
    )

## Model Training

Train all three sequence-to-sequence models with early stopping:

In [6]:
# Train models
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

histories = {}
for name, model in [('lstm', lstm_model), ('gru', gru_model), ('transformer', transformer_model)]:
    print(f"\nTraining {name.upper()} model:")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=32,
        callbacks=[early_stopping]
    )
    histories[name] = history

# Evaluate models
for name, model in [('lstm', lstm_model), ('gru', gru_model), ('transformer', transformer_model)]:
    loss, mse = model.evaluate(X_test, y_test)
    print(f"\n{name.upper()} Test Results:")
    print(f"Loss: {loss:.4f}")
    print(f"MSE: {mse:.4f}")


Training LSTM model:
Epoch 1/50
482/482 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - loss: 0.0310 - mse: 0.0282 - val_loss: 0.0107 - val_mse: 0.0101
Epoch 2/50
482/482 ━━━━━━━━━━━━━━━━━━━━ 21s 44ms/step - loss: 0.0091 - mse: 0.0086 - val_loss: 0.0089 - val_mse: 0.0083
Epoch 3/50
482/482 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - loss: 0.0085 - mse: 0.0080 - val_loss: 0.0086 - val_mse: 0.0081
Epoch 4/50
482/482 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - loss: 0.0081 - mse: 0.0076 - val_loss: 0.0083 - val_mse: 0.0078
Epoch 5/50
482/482 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - loss: 0.0079 - mse: 0.0074 - val_loss: 0.0085 - val_mse: 0.0079
Epoch 6/50
482/482 ━━━━━━━━━━━━━━━━━━━━ 21s 43ms/step - loss: 0.0078 - mse: 0.0073 - val_loss: 0.0085 - val_mse: 0.0080
Epoch 7/50
482/482 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - loss: 0.0076 - mse: 0.0071 - val_loss: 0.0078 - val_mse: 0.0073
Epoch 8/50
482/482 ━━━━━━━━━━━━━━━━━━━━ 22s 45ms/step - loss: 0.0074 - mse: 0.0069 - val_loss: 0.0074 - val_mse: 0.0070
Epoch 9/50
482/482

In [7]:
# Save models
lstm_model.save('../../models/pose_optimiser/lstm_model.keras')
gru_model.save('../../models/pose_optimiser/gru_model.keras')
transformer_model.save('../../models/pose_optimiser/transformer_model.keras')